In [61]:
import pandas as pd

In [62]:
query = "adbl"

# --- Load sentiment ---
sentiment_df = pd.read_csv(f"../DATA-HTML-STOCK/STOCKSENTIMENT/{query}_sentiment_results.csv", parse_dates=["Date"])
sentiment_df["Sentiment_Score"] = sentiment_df["Sentiment_Score"].astype(float)

print(sentiment_df.head())

        Date                                           Headline  \
0 2026-02-17  Bonus Shares of ADBL, ALBSL, SHIVM listed in N...   
1 2026-02-12  Falgun Interest Rate Update: Commercial Banks ...   
2 2026-01-14  Commercial Banks Revise Magh Interest Rates; H...   
3 2025-12-15  Commercial Banks Revise Poush Interest Rates; ...   
4 2025-12-11  NRB Report Highlights Commercial and Developme...   

                        Logit Prediction  Sentiment_Score  
0  [-0.8137, -1.4031, 2.6527]   positive         0.953698  
1  [-0.3563, -1.0445, 1.6315]   positive         0.829293  
2  [1.7010, -1.1284, -1.6185]   negative        -0.913054  
3  [1.5451, -0.8497, -1.6871]   negative        -0.884442  
4   [1.2508, -2.3492, 0.7846]   negative        -0.604346  


In [63]:
# Aggregate daily sentiment
daily_sentiment = sentiment_df.groupby("Date").agg(
    Avg_Sentiment=("Sentiment_Score", "mean")
).reset_index()


print(sentiment_df.head())

        Date                                           Headline  \
0 2026-02-17  Bonus Shares of ADBL, ALBSL, SHIVM listed in N...   
1 2026-02-12  Falgun Interest Rate Update: Commercial Banks ...   
2 2026-01-14  Commercial Banks Revise Magh Interest Rates; H...   
3 2025-12-15  Commercial Banks Revise Poush Interest Rates; ...   
4 2025-12-11  NRB Report Highlights Commercial and Developme...   

                        Logit Prediction  Sentiment_Score  
0  [-0.8137, -1.4031, 2.6527]   positive         0.953698  
1  [-0.3563, -1.0445, 1.6315]   positive         0.829293  
2  [1.7010, -1.1284, -1.6185]   negative        -0.913054  
3  [1.5451, -0.8497, -1.6871]   negative        -0.884442  
4   [1.2508, -2.3492, 0.7846]   negative        -0.604346  


In [64]:
# --- Load stock ---
stock_df = pd.read_csv(f"../DATA-HTML-STOCK/NEPSEDATA/{query}.csv", parse_dates=["Date"])

print(stock_df.head())

   S.N.       Date    Open    High     Low   Close  % Change        Qty  \
0     1 2026-02-05  295.00  298.50  292.90  296.00      0.30  19,054.00   
1     2 2026-02-04  293.00  299.00  293.00  295.10     -0.97  22,157.00   
2     3 2026-02-03  302.00  304.80  296.20  298.00     -0.33  10,054.00   
3     4 2026-02-02  304.00  304.00  296.00  299.00      0.30  17,660.00   
4     5 2026-02-01  302.00  304.00  297.00  298.10     -0.63  41,620.00   

        Turnover  
0   5,621,421.00  
1   6,558,735.50  
2   2,984,354.00  
3   5,261,467.50  
4  12,452,476.40  


In [65]:
# Keep only Close price, sort newest first
stock_daily = stock_df.groupby("Date").agg(Close=("Close", "last")).reset_index()
stock_daily = stock_daily.sort_values("Date", ascending=False).reset_index(drop=True)

print(stock_df.head())

   S.N.       Date    Open    High     Low   Close  % Change        Qty  \
0     1 2026-02-05  295.00  298.50  292.90  296.00      0.30  19,054.00   
1     2 2026-02-04  293.00  299.00  293.00  295.10     -0.97  22,157.00   
2     3 2026-02-03  302.00  304.80  296.20  298.00     -0.33  10,054.00   
3     4 2026-02-02  304.00  304.00  296.00  299.00      0.30  17,660.00   
4     5 2026-02-01  302.00  304.00  297.00  298.10     -0.63  41,620.00   

        Turnover  
0   5,621,421.00  
1   6,558,735.50  
2   2,984,354.00  
3   5,261,467.50  
4  12,452,476.40  


In [66]:
# --- Align news dates to trading dates ---
stock_dates = stock_daily["Date"].tolist()

def align_to_trading_day(news_date):
    future_dates = [d for d in stock_dates if d >= news_date]
    return future_dates[0] if future_dates else None

daily_sentiment["Date"] = daily_sentiment["Date"].apply(align_to_trading_day)
daily_sentiment = daily_sentiment.dropna(subset=["Date"])


In [67]:
# --- Merge stock + sentiment ---
merged_df = pd.merge(stock_daily, daily_sentiment, on="Date", how="left")
merged_df["Avg_Sentiment"] = merged_df["Avg_Sentiment"].fillna(0)


In [68]:
# --- Optional: rolling 3-day sentiment avg ---
merged_df["Sent_3day_avg"] = merged_df["Avg_Sentiment"].rolling(3, min_periods=1).mean()


In [69]:
# --- Target: next-day closing price ---
merged_df["Target_Close"] = merged_df["Close"].shift(-1)


In [70]:
# Drop last row (no target)
final_df = merged_df.dropna(subset=["Target_Close"]).reset_index(drop=True)


In [72]:
# Save final dataset
final_df.to_csv(f"{query}_final_dataset.csv", index=False)

print("Dataset ready! Sample:")
print(final_df)

Dataset ready! Sample:
           Date   Close  Avg_Sentiment  Sent_3day_avg Target_Close
0    2026-02-05  296.00       0.931026       0.931026       296.00
1    2026-02-05  296.00       0.946575       0.938800       296.00
2    2026-02-05  296.00       0.931835       0.936479       296.00
3    2026-02-05  296.00       0.943038       0.940482       296.00
4    2026-02-05  296.00       0.932877       0.935917       296.00
...         ...     ...            ...            ...          ...
3959 2010-09-13  116.00       0.000000       0.000000       118.00
3960 2010-09-12  118.00       0.000000       0.000000       122.00
3961 2010-09-09  122.00       0.000000       0.000000       125.00
3962 2010-09-08  125.00       0.000000       0.000000       138.00
3963 2010-09-07  138.00       0.000000       0.000000       255.00

[3964 rows x 5 columns]
